In [7]:
import pandas as pd
import re
from pathlib import Path

In [ ]:
# function to detect if there is intra-sentential code switching and sort words by language
def intraProcess(line, words, matrixLang, embedLang, switchCode, ambiWords):
    matrixLangWords = []
    embedLangWords = []
    for word in words:
        if re.search(r"(@s)", word):
            embedLangWords.append(word)
        else:
            matrixLangWords.append(word)
    if (len(matrixLangWords) == 0) or (len(embedLangWords) == 0):
        intraSwitch = False
        utteranceLang = embedLang
    elif set(matrixLangWords).issubset(ambiWords):
        intraSwitch = False
        utteranceLang = matrixLang
        matrixLangWords = words
        embedLangWords = []
    elif set(embedLangWords).issubset(ambiWords):
        intraSwitch = False
        utteranceLang = embedLang
        matrixLangWords = []
        embedLangWords = words
    else:
        intraSwitch = True
        if re.search(switchCode, line):
            utteranceLang = embedLang
            matrixLangWords, embedLangWords = embedLangWords, matrixLangWords
        else:
            utteranceLang = matrixLang
    matrixWordString = (" ").join(matrixLangWords).replace("@s", "")
    embedWordString = (" ").join(embedLangWords).replace("@s", "")

    return intraSwitch, utteranceLang, matrixWordString, embedWordString

In [ ]:
# Function to strip CHA markup and sort utterances into a dataframe
def parseToPkl(
    file,
    corpus,
    group,
    proficiency,
    targetFolder,
    matrixLang="english",
    embedLang="spanish/catalan",
    ambiWords=["hm", "mhm", "ah", "uhuh", "uhhuh", "eh", "huh", "oh", "no"],
):

    switchCode = (
        r"\[- " + embedLang[0:3] + "\]"
    )  # language code used to mark code-switched utterances
    fileName = Path(file).stem
    filePointer = open(file, "r", encoding="utf-8")
    fileContent = filePointer.read()
    fileLines = re.split(r"\s+(?=[\*])", fileContent)  # split file by utterances
    turnNo = 0
    utteranceNo = 0
    lastSpeaker = ""
    lastLang = ""
    prevSwitchDis = 0
    lastCS = False
    posInTurn = 1

    # headings for dataframe
    rowList = [
        [
            "UtteranceID",
            "FileID",
            "Corpus",
            "Group",
            "Proficiency",
            "Participant",
            "PrevPar",
            "UtteranceLang",
            "LangRole",
            "TurnNo",
            "UtteranceNo",
            "PosInTurn",
            "Utterance",
            "InterSwitch",
            "IntraSwitch",
            "PrevSwitchDistance",
            "LastCS",
            "MatrixLangWords",
            "EmbedLangWords",
            "Translation",
        ]
    ]

    for line in fileLines:

        if re.search(r"\*[A-Z]{3}:\t.*", line):  # line starts with *[XXX]:\t

            utterance = re.split(r"[\r\n]+(?=[%@])", line)[
                0
            ]  # take only the utterance if more context lines exist
            utterance = re.sub(
                r"[_+:]", "", utterance
            )  # remove + and _ from compound words
            words = re.findall(
                r"\b(?<![&=*@])(?!xxx)(?!www)[a-zA-ZÀ-ÿ\']+(?:@s)?(?![^\[]*\])",
                utterance,
            )  # gets list of all words in utterance, ignoring punctuation, anything in square brackets, and markup other than @s

            # ignore utterances that contain no words
            if len(words) < 1:
                continue

            utterance = (
                (" ").join(words).replace("@s", "")
            )  # join words into string and strip @s
            prevSwitchDis += 1  # increment distance from last inter-sentential switch

            if (set(words).issubset(ambiWords)) and (
                lastLang != ""
            ):  # check if utterance is made up of only ambiguous words
                utteranceLang = lastLang
                intraSwitch = False
                if utteranceLang == matrixLang:
                    matrixLangWords = utterance
                    embedLangWords = ""
                elif utteranceLang == embedLang:
                    embedLangWords = utterance
                    matrixLangWords = ""
            elif re.search(
                r"(@s)", line
            ):  # look for @s marker of intra-sentential code switching
                intraSwitch, utteranceLang, matrixLangWords, embedLangWords = (
                    intraProcess(
                        line, words, matrixLang, embedLang, switchCode, ambiWords
                    )
                )  # detect if true intra-switch and split words by language
            elif re.search(switchCode, line):
                intraSwitch = False
                utteranceLang = embedLang
                embedLangWords = utterance
                matrixLangWords = ""
            else:
                intraSwitch = False
                utteranceLang = matrixLang
                matrixLangWords = utterance
                embedLangWords = ""

            currentSpeaker = ((re.match(r"\*[A-Z]{3}:\t", line)).group())[
                1:4
            ]  # get 3-letter speaker reference
            if currentSpeaker != lastSpeaker:
                turnNo += 1
                posInTurn = 0
            posInTurn += 1
            utteranceNo += 1

            if (utteranceLang != lastLang) and (
                utteranceNo != 1
            ):  # if the utteranceLang is different from the last utterance, inter-CS is present
                interSwitch = True
            else:
                interSwitch = False

            if utteranceLang == matrixLang:
                langRole = "Matrix Language"
            else:
                langRole = "Embedded Language"

            if (intraSwitch == True) or (
                interSwitch == True
            ):  # if this utterance contains cs
                lastCS = True

            utteranceID = fileName + str(utteranceNo)

            trans = re.findall(
                r"(?:(?<=%eng:\t)).+", line
            )  # find translation string (starting with %eng in Miami Corpus)
            if trans:
                transWords = re.findall(
                    r"\b(?<![&=*@])[a-zA-ZÀ-ÿ\']+(?:@s)?(?![^\[]*\])", trans[0]
                )
                fullTranslation = (" ").join(transWords)
                row = [
                    utteranceID,
                    fileName,
                    corpus,
                    group,
                    proficiency,
                    currentSpeaker,
                    lastSpeaker,
                    utteranceLang,
                    langRole,
                    turnNo,
                    utteranceNo,
                    posInTurn,
                    utterance,
                    interSwitch,
                    intraSwitch,
                    prevSwitchDis,
                    lastCS,
                    matrixLangWords,
                    embedLangWords,
                    fullTranslation,
                ]
            else:
                row = [
                    utteranceID,
                    fileName,
                    corpus,
                    group,
                    proficiency,
                    currentSpeaker,
                    lastSpeaker,
                    utteranceLang,
                    langRole,
                    turnNo,
                    utteranceNo,
                    posInTurn,
                    utterance,
                    interSwitch,
                    intraSwitch,
                    prevSwitchDis,
                    lastCS,
                    matrixLangWords,
                    embedLangWords,
                ]

            rowList.append(row)

            lastSpeaker = currentSpeaker  # update last speaker
            lastLang = utteranceLang  # update last utterance language
            if intraSwitch == True:  # update if cs is present within utterance
                lastCS = True
            else:
                lastCS = False
            if (
                interSwitch == True
            ):  # update distance from prev switch if new switch is initiated
                prevSwitchDis = 0

    df = pd.DataFrame(rowList)
    df = df.rename(columns=df.iloc[0]).loc[1:]
    # print(targetFolder)
    Path(targetFolder).mkdir(parents=True, exist_ok=True)
    df.to_pickle(Path(targetFolder + fileName + ".pkl"))

<>:4: SyntaxWarning: invalid escape sequence '\]'
<>:4: SyntaxWarning: invalid escape sequence '\]'
C:\Users\szach\AppData\Local\Temp\ipykernel_2388\1364147866.py:4: SyntaxWarning: invalid escape sequence '\]'
  switchCode = r'\[- ' + embedLang[0:3] + '\]' # language code used to mark code-switched utterances


# BELC


In [10]:
# # parse group 1
# directory = Path('BELC/1-written(4t)_10-16')
# corpus = 'BELC'
# group = 'BELC 1'
# proficiency = 1
# targetFolder = './data/processed/new/BELC/1/'
# subdirectories = [f for f in directory.iterdir() if f.is_dir()]
# for subdirectory in subdirectories:
#     files = Path(subdirectory).glob('*.cha')
#     for file in files:
#         parseToPkl(file, corpus, group, proficiency, targetFolder)

In [11]:
# # parse group 2
# directory = Path('../data/raw/BELC/2/')
# corpus = 'BELC'
# group = 'BELC 2'
# proficiency = 2
# targetFolder = '../data/processed/new/BELC/2/'

# files = Path(directory).glob('*.cha')
# for file in files:
#     parseToPkl(file, corpus, group, proficiency, targetFolder)

In [12]:
# # parse group 3
# directory = Path('../data/raw/BELC/3/')
# corpus = 'BELC'
# group = 'BELC 3'
# proficiency = 3
# targetFolder = '../data/processed/new/BELC/3/'

# files = Path(directory).glob('*.cha')
# for file in files:
#     parseToPkl(file, corpus, group, proficiency, targetFolder)

In [13]:
# # parse group 4
# directory = Path('../data/raw/BELC/4/')
# corpus = 'BELC'
# group = 'BELC 4'
# proficiency = 4
# targetFolder = '../data/processed/new/BELC/4/'

# files = Path(directory).glob('*.cha')
# for file in files:
#     parseToPkl(file, corpus, group, proficiency, targetFolder)

In [ ]:
# Parse all dynamically

source_dir = Path("./data/raw/BELC/")
for task_dir in source_dir.iterdir():
    if task_dir.is_dir():
        sub_dirs = [task_dir]
        while sub_dirs:
            current_dir = sub_dirs.pop()
            for sub_dir in current_dir.iterdir():
                if sub_dir.is_dir():
                    sub_dirs.append(sub_dir)
                else:
                    if sub_dir.suffix == ".cha":
                        group = f"Group {sub_dir.stem[1]}"
                        proficiency = int(sub_dir.stem[0])
                        targetFolder = f"./data/processed/new/BELC/{task_dir.stem}/{current_dir.stem}/"
                        parseToPkl(sub_dir, "BELC", group, proficiency, targetFolder)

# Miami


In [ ]:
# parse eng ml
directory = Path("./data/raw/Miami/eng")
corpus = "Bangor Miami Corpus"
group = "Bangor Miami Corpus (eng)"
proficiency = 5
targetFolder = "./data/processed/new/Miami/eng/"

files = Path(directory).glob("*.cha")
for file in files:
    parseToPkl(file, corpus, group, proficiency, targetFolder, embedLang="spanish")

In [22]:
# parse spa ml
directory = Path("./data/raw/Miami/spa")
corpus = "Bangor Miami Corpus"
group = "Bangor Miami Corpus (spa)"
proficiency = 5
targetFolder = "./data/processed/new/Miami/spa/"

files = Path(directory).glob("*.cha")
for file in files:
    parseToPkl(
        file,
        corpus,
        group,
        proficiency,
        targetFolder,
        matrixLang="spanish",
        embedLang="english",
    )